<a href="https://colab.research.google.com/github/akfreeman72/Hotel-Demand-Forecasting-Project/blob/main/hannahisa444project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install statsforecast mlforecast neuralforecast lightgbm utilsforecast -q

In [ ]:
import pandas as pd

df = pd.read_parquet("sample_hotels-1.parquet")

print(df.head())

In [ ]:
import pandas as pd

df = pd.read_parquet("sample_hotels-1.parquet")
print(df.head())
print(df.columns)
print(df.shape)

In [ ]:
print(df.columns)
df.head()

In [ ]:
df = df[['unique_id', 'ds', 'y']].copy()
df = df.sort_values(['unique_id', 'ds'])

print(df.head())

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, AutoETS

sf_fast = StatsForecast(
    models=[
        Naive(),
        SeasonalNaive(season_length=7),
        AutoETS(season_length=7)
    ],
    freq='D',
    n_jobs=-1
)

cv_sf_fast = sf_fast.cross_validation(
    df=df,
    h=28,
    n_windows=5,
    step_size=28
)

cv_sf_fast.head()

In [ ]:
# AutoARIMA was too slow in this environment, so this model is being skipped temporarily.
# We completed Naive, SeasonalNaive, and AutoETS first.
cv_sf = cv_sf_fast.copy()

In [ ]:
cv_sf.to_csv("cross_validation_statsforecast.csv", index=False)

In [ ]:
df = df.groupby(["unique_id", "ds"], as_index=False)["y"].mean()

df = (
    df.set_index("ds")
      .groupby("unique_id")
      .resample("D")
      .asfreq()
      .drop(columns=["unique_id"])
      .reset_index()
)

df["y"] = df["y"].fillna(0)

df = df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

print(df.shape)
print(df.duplicated(subset=["unique_id", "ds"]).sum())

In [ ]:
cv_mf = mf.cross_validation(
    df=df,
    h=28,
    n_windows=5,
    step_size=28,
    refit=True
)

cv_mf.head()

In [ ]:
results = cv_sf.merge(
    cv_mf.drop(columns=["y"], errors="ignore"),
    on=["unique_id", "ds", "cutoff"],
    how="left"
)

results.head()

In [ ]:
results.to_csv("cross_validation_results.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd

def calculate_metrics(group):
    model_cols = [c for c in group.columns if c not in ["unique_id", "ds", "cutoff", "y"]]
    rows = []

    for model in model_cols:
        temp = group[["y", model]].dropna()
        error = temp[model] - temp["y"]

        me = error.mean()
        mae = error.abs().mean()
        rmse = np.sqrt((error ** 2).mean())

        nonzero = temp[temp["y"] != 0]
        mape = np.nan
        if len(nonzero) > 0:
            mape = (np.abs((nonzero["y"] - nonzero[model]) / nonzero["y"]).mean()) * 100

        rows.append({
            "unique_id": group["unique_id"].iloc[0],
            "Model": model,
            "ME": me,
            "MAE": mae,
            "RMSE": rmse,
            "MAPE": mape
        })

    return pd.DataFrame(rows)

final_metrics = (
    results
    .groupby("unique_id", group_keys=False)
    .apply(calculate_metrics)
    .reset_index(drop=True)
)

final_metrics.head()

In [ ]:
final_metrics.to_csv("metric_summary_by_series_method.csv", index=False)

In [ ]:
win_rows = []

for metric in ["MAE", "RMSE", "MAPE"]:
    metric_df = final_metrics.dropna(subset=[metric])
    winners = metric_df.loc[metric_df.groupby("unique_id")[metric].idxmin()]
    counts = winners["Model"].value_counts().reset_index()
    counts.columns = ["Model", "Wins"]
    counts["Metric"] = metric
    win_rows.append(counts)

me_df = final_metrics.copy()
me_df["abs_ME"] = me_df["ME"].abs()
me_winners = me_df.loc[me_df.groupby("unique_id")["abs_ME"].idxmin()]
me_counts = me_winners["Model"].value_counts().reset_index()
me_counts.columns = ["Model", "Wins"]
me_counts["Metric"] = "ME"
win_rows.append(me_counts)

win_counts = pd.concat(win_rows, ignore_index=True)

win_counts

In [ ]:
win_counts.to_csv("model_win_counts.csv", index=False)

In [ ]:
# Final forecasts using the full dataset

sf_forecast = sf_fast.forecast(df=df, h=28)

mf.fit(df)
mf_forecast = mf.predict(h=28)

final_forecasts = sf_forecast.merge(
    mf_forecast,
    on=["unique_id", "ds"],
    how="left"
)

final_forecasts.head()

In [ ]:
final_forecasts.to_csv("final_test_forecasts.csv", index=False)

In [ ]:
import os
import matplotlib.pyplot as plt

os.makedirs("plots", exist_ok=True)

model_cols = [c for c in final_forecasts.columns if c not in ["unique_id", "ds"]]

for uid in df["unique_id"].unique():
    hist = df[df["unique_id"] == uid].tail(120)
    future = final_forecasts[final_forecasts["unique_id"] == uid]

    plt.figure(figsize=(12, 5))
    plt.plot(hist["ds"], hist["y"], label="Actual", linewidth=2)

    for model in model_cols:
        plt.plot(future["ds"], future[model], label=model, linestyle="--")

    plt.title(f"28-Day Forecast vs Actual History: {uid}")
    plt.xlabel("Date")
    plt.ylabel("Room Demand")
    plt.legend()
    plt.tight_layout()

    plt.savefig(f"plots/{uid}_forecast.png", dpi=300)
    plt.show()

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.auto import AutoNBEATS, AutoNHITS

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.auto import AutoNBEATS, AutoNHITS

nbeats_config = {
    "input_size": 56,
    "max_steps": 50
}

nhits_config = {
    "input_size": 56,
    "max_steps": 50
}

nf = NeuralForecast(
    models=[
        AutoNBEATS(h=28, config=nbeats_config, num_samples=3),
        AutoNHITS(h=28, config=nhits_config, num_samples=3)
    ],
    freq="D"
)

In [ ]:
cv_nf = nf.cross_validation(
    df=df,
    h=28,
    n_windows=5,
    step_size=28
)

cv_nf.head()

In [ ]:
cv_nf.head()
cv_nf.columns

In [ ]:
results = results.merge(
    cv_nf.drop(columns=["y"], errors="ignore"),
    on=["unique_id", "ds", "cutoff"],
    how="left"
)

results.to_csv("cross_validation_results.csv", index=False)

results.head()

In [ ]:
print(cv_nf.head())
print(cv_nf.columns)

In [ ]:
results = results.merge(
    cv_nf.drop(columns=["y"], errors="ignore"),
    on=["unique_id", "ds", "cutoff"],
    how="left"
)

results.to_csv("cross_validation_results.csv", index=False)

In [ ]:
final_metrics
win_counts

In [ ]:
final_metrics.to_csv("metric_summary_by_series_method.csv", index=False)
win_counts.to_csv("model_win_counts.csv", index=False)

In [ ]:
nf.fit(df)
nf_forecast = nf.predict()

final_forecasts = final_forecasts.merge(
    nf_forecast,
    on=["unique_id", "ds"],
    how="left"
)

final_forecasts.to_csv("final_test_forecasts.csv", index=False)